## Lab07: RAG (Retrieval-Augmented-Generation)

RAG (Retrieval-Augmented Generation) is a powerful framework in Natural Language Processing (NLP) that enhances the performance of language models by combining traditional generative models with external knowledge retrieval. This hybrid approach allows models to retrieve relevant information from a large corpus (like a database or document collection) and incorporate this information into the generation process. It is particularly useful when a model needs to answer questions, generate content, or provide explanations based on real-time or domain-specific data.



In [ ]:
# First, let's import some useful functions from the previous lab


import torch

#detect the device available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device: ", device) # You may need to execute "pip install torch --index-url https://download.pytorch.org/whl/cu124" in order to use the GPU

In [ ]:

from transformers import AutoTokenizer, AutoModelForCausalLM, GenerationConfig

# HuggingFace token

#----------------Google Colab only---------------------
# from huggingface_hub import login
#from google.colab import userdata
#login(userdata.get('HF_TOKEN'))
#----------------Google Colab only---------------------

#----------------Local Runtime only---------------------
import os
from dotenv import load_dotenv # pip install python-dotenv
from huggingface_hub import login
load_dotenv()
hf_token = os.getenv("HF_TOKEN") # Create this constant in the file .env (put it in the root dir of your project)
login(token=hf_token)
#----------------Local Runtime only---------------------


# Load the model and the tokenizer (the first time it requires a lot of time to be executed)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_id = "meta-llama/Llama-3.2-3B-Instruct"
model = AutoModelForCausalLM.from_pretrained(model_id, dtype=torch.float16, device_map=device, token=hf_token)
tokenizer = AutoTokenizer.from_pretrained(model_id, token=hf_token, clean_up_tokenization_spaces=False)


# Configure generation parameters
gen_config = GenerationConfig.from_model_config(model.config)
# gen_config.temperature = 0.3              # Regulate randomness, use with do_sample = True
# gen_config.max_length = None              # Since max_new_tokens is already set
gen_config.repetition_penalty = 2.0         # Penalize repetition
gen_config.no_repeat_ngram_size = 3         # Avoid repeating bigrams
gen_config.do_sample = False                # Set to False in order to use Greedy or Beam search
gen_config.num_beams = 3                    # Use with do_sample = False
gen_config.eos_token_id = tokenizer.eos_token_id  # End generation at EOS token
gen_config.pad_token_id = tokenizer.pad_token_id
gen_config.early_stopping = True



#alternative 1: use the tokenizer and produce the output

def make_a_query(_prompt: str, _tokenizer, _model, _device, _gen_config, max_new_tokens:int = 200):

    # Set pad_token_id if missing
    if not hasattr(_tokenizer, "pad_token_id") or _tokenizer.pad_token_id is None:
        _tokenizer.pad_token_id = _tokenizer.eos_token_id

    _gen_config.max_new_tokens = max_new_tokens  # Limit the number of new tokens generated (e.g., a single word)


    # Tokenize the input with padding and truncation
    inputs = _tokenizer(_prompt, return_tensors="pt", padding=True, truncation=True).to(_device)

    # Compute the length of the input prompt to be able to extract the model's response later
    input_ids = inputs["input_ids"]
    prompt_length = input_ids.shape[1]

    # Generate a response
    output = _model.generate(
        input_ids,
        attention_mask=inputs["attention_mask"],
        generation_config=_gen_config,
    )

    generated_tokens = output[0, prompt_length:]    # Try to print the variable "output" in order to understand its dimensionality

    # Decode the response into human-readable text
    response = _tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

    return response

In [ ]:
import pymupdf

def extract_text_from_pdf(pdf_path):
    doc = pymupdf.open(pdf_path)
    text = ""
    for page_num in range(len(doc)):
        page = doc.load_page(page_num)
        text+=page.get_text()
    return text

In [ ]:
import os
import glob


# Extract text from all PDF files
pdf_texts = {}
directory="."
pdf_files = glob.glob(os.path.join(directory,"paper*.pdf"))
n = 0
for pdf_file in pdf_files:
    pdf_texts[n] = extract_text_from_pdf(pdf_file)
    n = n+1

#Display the text from all the PDF files
for pdf_file, txt in pdf_texts.items():
    print(f"--- {pdf_file} ---")
    print(txt[:500])  # Display the first 500 characters of each document
    print("\n")

### Creating an index of vectors to represent the documents

To perform efficient searches, we need to convert our text data into numerical vectors. To do so, we will use the first step of the BERT transformer.

Since our full pdf files are very long to be fed as input into BERT, we perform a step in which we create a structure where we associate a document number to its abstract, and in a separate dictionary we associate a document number to its full text.


In [ ]:
from transformers import AutoModel, AutoTokenizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np




#import the Bert pretrained model from the transformers library
model_embeddings = AutoModel.from_pretrained("bert-base-uncased")
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
### Don't forget to move it to the device

#initialization of the dictionary of abstracts. Substitute this with the abstracts of the 10 papers considered as sources for RAG
#(we could use functions to read the PDFs to "cut" the abstracts from the papers. For simplicity reasons, we will copy and paste them)
abstracts_dict = {
    0: ""
}

#the text for rag is used as an input to the BERT model

#The tokenized inputs are passed to the BERT model for processing.
#(#remember padding=True: Ensures that all inputs are padded to the same length, allowing batch processing.)
#The model outputs a tensor (last_hidden_state), where each input token is represented by a high-dimensional vector.
#last_hidden_state is of shape (batch_size, sequence_length, hidden_size), where:
#batch_size: Number of input texts.
#sequence_length: Length of each tokenized text (after padding).
#hidden_size: Dimensionality of the vector representation for each token (default 768 for bert-base-uncased).

#last_hidden_state[:, 0]: Selects the representation of the [CLS] token for each input text. The [CLS] token is a special token added at the start of each input and is often used as the aggregate representation for the entire sequence.

# -------------------------------------- HINT -----------------------------------
tokenized_inputs = tokenizer(
    list(abstracts_dict.values()),
    return_tensors='pt', # stands for 'PyTorch': tensors will be returned (if omitted, we will get Python lists)
    padding=True)   # all the abstracts will be padded to the same length, in order to have a uniformed batch of abstracts => an attention mask will be created
tokenized_inputs = {key: value.to(device) for key, value in tokenized_inputs.items()}   # move the tensors to the device

abstract_vectors = model_embeddings(**tokenized_inputs).last_hidden_state[:, 0]

#abstract_vectors is a tensor of shape (batch_size, hidden_size) (e.g., (3, 768) in this case), representing each text as a single 768-dimensional vector.

print(abstract_vectors.shape)



### Search

With our text data vectorized and indexed, we can now perform searches. We will define a function to search the index for the most relevant documents based on a query.

To perform the search, we need a function (search documents) where we perform the cosine similarity between the query vector and all the abstract vectors. This function will give our the top-k indexes. Once we find the top-k indexes, with another function, we can collect the full text of the documents from the paper dictionary.

To compute cosine similarity, refer to the following formula

```cs = cosine_similarity(vector_a.detach().numpy(), vector_b.detach().numpy())```



In [ ]:




def get_top_k_similar_indices(query_vector,         # i.e., the QUERY in the RAG system
                              abstract_vectors,     # i.e., the KEY in the RAG system
                              k):

    #Parameters:
    #- query_vector: A tensor of shape (1, hidden_size) representing the query vector.
    #- abstract_vectors: A tensor of shape (batch_size, hidden_size) representing the vector embeddings of the abstract.
    #- k: The number of top indices to return.

    #Returns:
    #- sorted_indices: A numpy array of shape (1, k) containing the indices of the top k most similar abstracts.

    # ------------- HINT ----------------------------------------------------------------
    #Computes the top k indices of the most similar abstracts to the query based on cosine similarity.
    similarities = cosine_similarity(   # from sklearn.metrics.pairwise
        query_vector.cpu().detach().numpy(),    # it is needed to move the tensor on the cpu!
        abstract_vectors.cpu().detach().numpy())

    # IMPORTANT: reason about the size of 'similarities' to understand how to sort it (you can print it)
    # ------------- HINT ----------------------------------------------------------------

    return ""


def retrieve_documents(indices, documents_dict):
    """
    Retrieves the documents corresponding to the given indices and concatenates them into a single string.
    :param indices: A numpy array or list of top-k indices of the most similar documents.
    :param documents_dict: A dictionary where keys are document indices (integers) and values are the document texts (strings).
    :return: concatenated_documents: A string containing the concatenated texts of the retrieved documents.
    """

    return ""



# Now, create a vector embedding also for the query

# 1) from NL to tokens

query = ""  # remember to move the tokenised input on the correct device (the same as the model)

# 2) from tokens to vector (i.e., embedding)

query_vector = ""

# TODO: get the top k abstracts similar to the query ( call get_top_k_similar_indices() )

# TODO: get the relative texts ( call retrieve_documents() )





### A function to perform Retrieval Augmented Generation

In this step, we’ll combine the context retrieved from our documents with LLAMA to generate responses. The context will provide the necessary information to the model to produce more accurate and relevant answers.

In [ ]:


#now we put it all together

def generate_augmented_response(query, documents):

    system = ""             #TODO: define system prompt

    context = ""               #TODO: concatenate here all the search results


    prompt = ""                 #TODO: create the prompt for LLAMA (system + context + query)

    response = ""

    #perform a query with LLAMA in the usual way ( call make_a_query() )

    #return the response
    return ""


# TODO: generate the queries!
query = ""
response = generate_augmented_response(query)
print(response)

#TODO: now compare the results with a prompt without RAG. What are the results?
